# <strong>Knowledge Graph Embeddings</strong>

We generate embedding for the Knowledge graph create earlier

To make things easier we refer to the json file created, rather then quering from neo4j api where our Knowledge graph is stored

<img src="Images\KnowledgeGraphEmbedding.png" width=600px>

above diagram show how we will use the Knowledge graph embeddings to our Model

#### Imports

In [ ]:
# we need to make a list of triplets for getting embeddings
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import json
import pykeen
import os
import numpy as np
import torch
import logging

torch.manual_seed(43)

# Set the logging level to ERROR
logging.getLogger("pykeen").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

<hr>

In [ ]:
# set file paths
bankrupt_json_path = r'output\bankrupt'
healthy_json_path = r'output\healthy'

bankrupt_files_triplets = []
healthy_files_triplets = []

In [ ]:
torch.cuda.is_available()

<hr>

##### Create **Entity2id** and **Relation2id** dict to effectivly number and convert any entitiy into numerical data

In [ ]:
bankrupt_json_file = os.listdir(bankrupt_json_path)
healthy_json_file = os.listdir(healthy_json_path)

print(f'Found {bankrupt_json_file.__len__()} bankrupt company json files')
print(f'Found {healthy_json_file.__len__()} healthy company json files')

# initialize dictionaries for entities and relations to ids
entity2id = defaultdict(lambda: len(entity2id))
relation2id = defaultdict(lambda: len(relation2id))

This function will generate Embeddings by making use of pretrained TransE model for Knowledge graph Embeddings

In [ ]:
def generateKGE(json_file, bankrupt):
    """
    Generates (30, 0) embeddings for each company using pykeen library
    """
    ent_emb = dict()
    rel_emb = dict()
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        triples = []
        for every_relation in data["relations"]:
            entity2id[every_relation["source"]]
            relation2id[every_relation["relation"]]
            entity2id[every_relation["target"]]
            triples.append((every_relation["source"], every_relation["relation"], every_relation["target"]))

        triples_factory = TriplesFactory.from_labeled_triples(np.array(triples))
        result = pipeline(
            model="TransE",
            dataset=None,
            training=triples_factory,
            testing=triples_factory,
            model_kwargs=dict(embedding_dim=10),
        )

        for i, content in enumerate(result.model.entity_representations[0]().detach().numpy()):
            ent_emb[triples_factory.entity_labeling.id_to_label[i]] = content
            
        for i, content in enumerate(result.model.relation_representations[0]().detach().numpy()):
            rel_emb[triples_factory.relation_labeling.id_to_label[i]] = content
        
        def aggregate():
                store = [np.concatenate(
                     [
                        ent_emb[s],
                        rel_emb[r],
                        ent_emb[t]
                     ]
                ) for s, r, t in triples]
                return np.mean(store, axis=0)
        company_embedding = aggregate()
        return [company_embedding, 1 if bankrupt else 0]

In [ ]:
##############################################################
# 16/12/24 | 11:15 AM | 
# Aggregation of embeddings is done by taking mean of all the embeddings
# each triples creates one embeddings, these embeddings are then averaged
#****IMPROVMENTS***###########################################
# 1. Add more models.
# 2. Proper test-train split
##############################################################
import json
import numpy as np
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

all_json_files = []

for file in bankrupt_json_file:
    all_json_files.append(os.path.join(bankrupt_json_path, file))

for file in healthy_json_file:
    all_json_files.append(os.path.join(healthy_json_path, file))

def train_model_once(files, model="TransE", embedding_dim=10, split_ratio=[0.8, 0.2]):
    """
    Train the model on a unified dataset from multiple files.
    Returns the trained model and entity/relation mappings.
    """
    all_triples = []

    # Combine triples from all files
    for json_file in files:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for relation in data["relations"]:
                entity2id[relation["source"]]
                relation2id[relation["relation"]]
                entity2id[relation["target"]]
                all_triples.append(
                    (relation["source"], relation["relation"], relation["target"])
                )

    # Create a unified triples factory
    triples_factory = TriplesFactory.from_labeled_triples(np.array(all_triples))

    # Split into training and testing
    try:
        training_factory, testing_factory = triples_factory.split(split_ratio)
    except Exception as e:
        training_factory = triples_factory
        testing_factory = triples_factory

    # Train the model
    result = pipeline(
        model=model,
        dataset=None,
        training=training_factory,
        testing=testing_factory,
        model_kwargs=dict(embedding_dim=embedding_dim),
    )

    return result.model, triples_factory

def generate_embeddings_for_file(json_file, model, triples_factory):
    """
    Generates embeddings for a single file using a pre-trained model.
    """
    ent_emb = {}
    rel_emb = {}

    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        triples = []
        for relation in data["relations"]:
            triples.append((relation["source"], relation["relation"], relation["target"]))

    # Extract entity and relation embeddings
    for i, content in enumerate(model.entity_representations[0]().detach().numpy()):
        ent_emb[triples_factory.entity_labeling.id_to_label[i]] = content

    for i, content in enumerate(model.relation_representations[0]().detach().numpy()):
        rel_emb[triples_factory.relation_labeling.id_to_label[i]] = content

    # Aggregate embeddings
    def aggregate():
        store = [
            np.concatenate([
                ent_emb[s],
                rel_emb[r],
                ent_emb[t]
            ]) for s, r, t in triples
        ]
        return np.mean(store, axis=0)

    company_embedding = aggregate()
    return company_embedding

model_te, triple_factory_te = train_model_once(all_json_files)
model_re, triple_factory_re = train_model_once(all_json_files, model="RotatE")
##############################################################

In [ ]:
all_company_data = []

for files in bankrupt_json_file:
    all_company_data.append(generateKGE(os.path.join(bankrupt_json_path, files), True))

for files in healthy_json_file:
    all_company_data.append(generateKGE(os.path.join(healthy_json_path, files), False))


In [ ]:
assert len(all_company_data) == bankrupt_json_file.__len__() + healthy_json_file.__len__()

In [ ]:
##############################################################
# 16/12/24 | 10:15 PM | 
# generating embeddings for all companies
# files are in order so the labels are 50 bankrupt [1] and 50 healthy 
##############################################################
all_company_data_te = []

for files in all_json_files:
    all_company_data_te.append(generate_embeddings_for_file(files, model_te, triple_factory_te))

label_te = [50*[1] + 50*[0]]

all_company_data_re = []

for files in all_json_files:
    all_company_data_re.append(generate_embeddings_for_file(files, model_re, triple_factory_re))

label_re = [50*[1] + 50*[0]]

assert len(all_company_data_te) == bankrupt_json_file.__len__() + healthy_json_file.__len__()
assert len(all_company_data_re) == bankrupt_json_file.__len__() + healthy_json_file.__len__()
##############################################################

In [ ]:
##############################################################
# 16/12/24 | 10:15 PM | 
# use these embeddings as dataset for training NN (adding NN before LR)
##############################################################
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
normalized_embedding = scaler.fit_transform(all_company_data_te)

X_te = np.array(normalized_embedding)
y_te = np.array(label_te).flatten()

X_re = np.array(all_company_data_re)
y_re = np.array(label_re).flatten()
##############################################################

In [ ]:
company_embeddings = []
labels = []

for company_data in all_company_data:
    company_embeddings.append(company_data[0])
    labels.append(company_data[1])

X = np.array(company_embeddings)
y = np.array(labels)

It is crucial to map the input file and its embedding to match up them at later stage

In [ ]:
mapping_dict = {}
for file, x in zip(bankrupt_json_file, X[:50]):
    mapping_dict[file] = x

for file, x in zip(healthy_json_file, X[51:]):
    mapping_dict[file] = x

mapping_dict

In [ ]:
#####################################################################
# 16/12/24 | 11:15 PM |
# Mapping dict to their models 
# looks like rotatE uses complex number too!
#####################################################################
mapping_dict_te = {}
for file, x in zip(bankrupt_json_file, X_te[:50]):
    mapping_dict_te[file] = x

for file, x in zip(healthy_json_file, X_te[51:]):
    mapping_dict_te[file] = x


mapping_dict_re = {}
for file, x in zip(bankrupt_json_file, X_re[:50]):
    mapping_dict_re[file] = x

for file, x in zip(healthy_json_file, X_re[51:]):
    mapping_dict_re[file] = x
#####################################################################

In [ ]:
for i in mapping_dict_te.items():
    print(i, i[1].shape)
    break

for i in mapping_dict_re.items():
    print(i, i[1].shape)
    break

In [ ]:
# convert numpy array to list
for key, value in mapping_dict.items():
    mapping_dict[key] = value.tolist()

In [ ]:
# save the embeddings to a json file
with open('output/embeddings.json', 'w') as f:
    json.dump(mapping_dict, f)


In [ ]:
#####################################################################
# 16/12/24 | 11:15 PM |
# Mapping dict to their models 
# looks like rotatE uses complex number!
# PROBLEMS ##########################################################
# 1. RotatE embeddings are complex numbers
# 2. Connot convert complex numbers to json
# SOLUTION ##########################################################
# 1. Convert complex numbers to string and while using convert back using complex()
#####################################################################
for key, value in mapping_dict_te.items():
    mapping_dict_te[key] = value.tolist()

for key, value in mapping_dict_re.items():
    mapping_dict_re[key] = [str(cmplx) for cmplx in value.tolist()]

with open('output/embeddings_te.json', 'w') as f:
    json.dump(mapping_dict_te, f)

with open('output/embeddings_re.json', 'w') as f:
    json.dump(mapping_dict_re, f)
#####################################################################

In [ ]:
########################################################
# 16/12/24 | 11:43 PM 
# Saving transE model with 30 embeddings
########################################################
# convert into a csv file, easier to read
import pandas as pd
df = pd.DataFrame.from_dict(mapping_dict_te, orient='index',columns=[f"k{i}" for i in range(1, 31)])
df.index.name = "path"
df.to_csv('output/embeddings_te_30.csv')
########################################################

### Classification test

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, y_train = X, y

In [ ]:
clf = LogisticRegression(random_state=0)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_train)
print(f'Accuracy: {accuracy_score(y_train, y_pred)}')
print(classification_report(y_train, y_pred))

In [ ]:
#####################################################################
# 16/12/24 | 11:15 PM |
# PROBLEMS ##########################################################
# 1. RotatE embeddings are complex numbers
# 2. Model cannot work with complex numbers
#####################################################################
X_train, X_test, y_train, y_test = train_test_split(X_te, y_te, test_size=0.2, random_state=42, shuffle=True)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle=True)
clf_te = LogisticRegression(random_state=0)
clf_te.fit(X_train, y_train)

# X_re_str = np.array([[str(x) for x in row] for row in X_re])
# clf_re = LogisticRegression(random_state=0)
# clf_re.fit(X_re_str, y_re)
#####################################################################

<hr>

In [ ]:
y_pred = clf_te.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# saving the model 
import pickle
with open('output/logistic_regression_te.pkl', 'wb') as f:
    pickle.dump(clf_te, f)
print("Model saved successfully")    

In [ ]:
with open('output/logistic_regression_te.pkl', 'rb') as f:
    clf_te = pickle.load(f)

for i in range(X_val.shape[0]):
    y_pred = clf_te.predict(X_val[i].reshape(1, -1))
    print(y_pred, y_val[i])

<hr>

#### **Neural Network**

created a neural network for binary classification task to classify KGE into 2 classes, bankrupt and healthy, by this we map the KGE even closer to embeddings of Bankrupt and Healthy company files

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class EmbeddingModel(nn.Module):
    def __init__(self, input_dim=30, embedding_dim=10):
        super(EmbeddingModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.ReLU(),
            nn.BatchNorm1d(100),

            nn.Linear(100, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),

            nn.Linear(32, embedding_dim),
            nn.ReLU(),

        )

        self.classifier = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        embeddings = self.model(x)
        output = torch.sigmoid(self.classifier(embeddings))
        return embeddings, output
    
model = EmbeddingModel()

##### Train the model to fit for binary classification

In [ ]:
X_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(y, dtype=torch.float32)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    embeddings, outputs = model(X_train)
    loss = criterion(outputs, y_train.view(-1, 1))
    loss.backward()
    optimizer.step()

    if(epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch+1}, Loss: {loss.item():.4f}')

model.eval()
with torch.no_grad():
    X_train_embeddings, _ = model(X_train)


In [ ]:
######################################################
# 17/12/24 | 12:00 AM
# NN model for refining embeddings 
# now the model is loading properly
######################################################
X_train = torch.tensor(X_te, dtype=torch.float32)
y_train = torch.tensor(y_te, dtype=torch.float32)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    embeddings, outputs = model(X_train)
    loss = criterion(outputs, y_train.view(-1, 1))
    loss.backward()
    optimizer.step()

    if(epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch+1}, Loss: {loss.item():.4f}')

model.eval()
with torch.no_grad():
    X_train_embeddings, _ = model(X_train)

torch.save(model.state_dict(), "binary_classifier_nn.pth")
print("Model saved successfully")
######################################################

In [ ]:
model = EmbeddingModel()
model.load_state_dict(torch.load("binary_classifier_nn.pth"))
model.eval()
with torch.no_grad():
    X_train_embeddings, _ = model(X_train)

In [ ]:
######################################################
# 17/12/24 | 12:30 AM
# store the embeddings in a json file with shape (10,)
# scaling : NO : embeddings became 0 after scaling
######################################################
mapping_dict_te_10 = {}

for file, x in zip(bankrupt_json_file, X_train[:50]):
    mapping_dict_te_10[file], _ = model(x.unsqueeze(0))
for file, x in zip(healthy_json_file, X_train[51:]):
    mapping_dict_te_10[file], _ = model(x.unsqueeze(0))

for key, value in mapping_dict_te_10.items():
    mapping_dict_te_10[key] = value.tolist()[0]
    
# convert into a csv file, easier to read
import pandas as pd
df = pd.DataFrame.from_dict(mapping_dict_te_10, orient='index',columns=["k1", "k2", "k3", "k4", "k5", "k6", "k7", "k8", "k9", "k10"])
df.index.name = "path"
df.to_csv('output/embeddings_te_10.csv')
######################################################


In [ ]:
mapping_dict = {}

for file, x in zip(bankrupt_json_file, X_train[:50]):
    mapping_dict[file], _ = model(x.unsqueeze(0))
for file, x in zip(healthy_json_file, X_train[51:]):
    mapping_dict[file], _ = model(x.unsqueeze(0))

for key, value in mapping_dict.items():
    mapping_dict[key] = value.tolist()[0]

Store the embeddings in json file or in csv file for easy retrival!

In [ ]:
# convert into a csv file, easier to read
import pandas as pd
df = pd.DataFrame.from_dict(mapping_dict, orient='index',columns=["k1", "k2", "k3", "k4", "k5", "k6", "k7", "k8", "k9", "k10"])
df.index.name = "path"
df.to_csv('output/embeddings.csv')

In [ ]:
# converting it into json file
with open('output/embeddings.json', 'w') as f:
    json.dump(mapping_dict, f)

<hr>
<hr>

##### Rough / Sample trials

In [ ]:
# testing it on single file
# source, target, relation
one_file = []
with open(os.path.join(bankrupt_json_path, bankrupt_json_file[2]), 'r') as f:
    jsonfile = json.load(f)
    for every_relation in jsonfile['relations']:
        one_file.append((every_relation['source'], every_relation['relation'], every_relation['target']))

entity2id = defaultdict(lambda: len(entity2id))
relation2id = defaultdict(lambda: len(relation2id))

triplet_ids = [[(entity2id[s], relation2id[r], entity2id[o]) for s, r, o in one_file], 1]
print(one_file)
print(triplet_ids[0])
triple_factory = TriplesFactory.from_labeled_triples(np.array(one_file))
print(triple_factory)
print(triple_factory.entity_labeling)

result = pipeline(
    model="TransE",
    dataset=None,
    training=triple_factory,
    testing=triple_factory,
    model_kwargs=dict(embedding_dim=10)
)

In [ ]:
ent_emb = dict()
rel_emb = dict()
for i, content in enumerate(result.model.entity_representations[0]().detach().numpy()):
    ent_emb[triple_factory.entity_labeling.id_to_label[i]] = content
    
for i, content in enumerate(result.model.relation_representations[0]().detach().numpy()):
    rel_emb[triple_factory.relation_labeling.id_to_label[i]] = content

print(ent_emb)
print(rel_emb)

def func():
    k = [
        np.concatenate(
            [
                ent_emb[s],
                rel_emb[r],
                ent_emb[t]
            ]
        )
        for s, r, t in one_file
    ]
    kk = np.mean(k, axis=0)
    return kk
print(func().shape)

In [ ]:
# use this to access the correct row from embedding matrix
triple_factory.entity_labeling.label_to_id['Adhunik Metaliks Limited']

In [ ]:


def get_triplets(json_file, bankrupt):
    triples = []
    with open(json_file) as f:
        jsonfile = json.load(f)
        for every_relation in jsonfile['relations']:
            triples.append((every_relation['source'], every_relation['relation'], every_relation['target']))
    return [triples, 1 if bankrupt else 0]


def generate_kg_embeddings(triplets):
    training_data = [(str(s), str(o), str(r)) for s, r, o in triplets]
    triples_factory = TriplesFactory.from_labeled_triples(np.array(training_data))

    result = pipeline(
        model="TransE",
        dataset=None,
        training=triples_factory,
        testing=triples_factory,
        training_kwargs=dict(num_epochs=100),
        model_kwargs=dict(embedding_dim=10),
    )

    entity_embeddings = result.model.entity_representations[0]().detach().numpy()
    relation_embeddings = result.model.relation_representations[0]().detach().numpy()
    
    return entity_embeddings, relation_embeddings, result

def aggregate_comapny_embeddings(triplet_ids, entity_embeddings, relation_embeddings):
    # aggregate multiple embeddings into single triplet embedding

    triplet_embeddings = [
        np.concatenate(
            [
                entity_embeddings[head],
                relation_embeddings[relation],
                entity_embeddings[tail]
            ]
            )
            for head, relation, tail in triplet_ids
    ]

    company_embedding = np.mean(triplet_embeddings, axis=0)
    return company_embedding

In [ ]:
# extract triplets from json files
all_company_data = []
for file in bankrupt_json_file:
    s, r, t = get_triplets(os.path.join(bankrupt_json_path, file), True)
    entity2id[s], relation2id[r], entity2id[t]
    all_company_data.append((s, r, t))

for file in healthy_json_file:
    s, r, t = get_triplets(os.path.join(healthy_json_path, file), False)
    entitiy2id[s], relation2id[r], entitiy2id[t]
    all_company_data.append((s, r, t))

assert len(all_company_data) == bankrupt_json_file.__len__() + healthy_json_file.__len__()

In [ ]:
for company_data in all_company_data:
    company_triplets, label = company_data
    
    entity_emb, relation_emb = generate_kg_embeddings(company_triplets)

    triplets_id = [(entitiy2id(s), relation2id(r), entitiy2id(t)) for s, r, t, in company_triplets]

    

    

In [ ]:
# Example company triplets (convert these for each company as needed)
triplets_for_company = triplet_ids  # Using the previously converted triplet IDs

# Generate embeddings
entity_emb, relation_emb = generate_kg_embeddings(triplets_for_company)

# Compute the final embedding for the company by averaging the triplet embeddings
company_embedding = aggregate_comapny_embeddings(triplets_for_company, entity_emb, relation_emb)


In [ ]:
company_embedding.shape